# Environment Setup

In [41]:
! pip install --upgrade google-cloud-documentai

In [42]:
! pip install tabulate

In [43]:
! pip install Pillow

In [44]:
# import
import os
import pandas as pd
from IPython.display import display
from tabulate import tabulate

# Google Document API Setup

In [45]:
# Set environment variable
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../../src/utils/../../src/utils/service_account_credentials.json"
print("Credentials set succesfully.")

Credentials set succesfully.


In [46]:
from google.cloud import documentai_v1 as documentai

project_id = "193893511134"         # Replace with your GCP project ID
location = "us"                        # Or "eu" depending on where your processor is
processor_id = "e2ef8ddeb2ead929"     # Replace with your Document AI processor ID

client = documentai.DocumentProcessorServiceClient()

# Full resource name of the processor
processor_name = f"projects/{project_id}/locations/{location}/processors/{processor_id}"


# Load Dataset -> Trainset

In [7]:
# trainset

cloud_directory="F:/Axiler25/Research2/Datasets/reduce-latest-cloud/"
device_directory="F:/Axiler25/Research2/Datasets/reduce-latest-device/"

cloud_train_directory=os.path.join(cloud_directory, "reduce-latest-train-multimodal-cloud.csv")
device_train_directory=os.path.join(device_directory, "reduce-latest-train-multimodal-device.csv")
# Read the CSV files
cloud_train=pd.read_csv(cloud_train_directory)
device_train=pd.read_csv(device_train_directory)

In [8]:
cloud_train.drop(columns=["Unnamed: 0", "Unnamed: 0.1"],inplace=True)

In [9]:
device_train.drop(columns=["Unnamed: 0"], inplace=True)

In [10]:
# cloud sample
cloud_train.head(1)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,I am using Azure Devops to build and push my D...,\n \nI have created a pipeline ...,"['docker', 'azure-devops', 'continuous-integra...",https://stackoverflow.com/questions/60287354/i...,41,2020-02-18T18:32:17,2,['This is exactly what I wanted. Thanks for sh...,[{'body': '\nWhen using buildAndPush command i...,\nWhen using buildAndPush command in Docker ta...,['https://i.sstatic.net/D7pQd.png']


In [11]:
# device sample
device_train.head(1)

,title,body,tags,link,score,creation_date,answer_count,comments,answers,selected_answer,images
0,Outgrowing Linksys WRT54G router [closed],\n \n \n ...,"['router', 'firewall', 'router', 'firewall']",https://superuser.com/questions/48653/outgrowi...,5,2009-09-29 18:42:53Z,8,"[""Right now I'm looking at sonicwall but I don...",[{'body': '\nNetgear WNDR3700\n\n\nIEEE 802.11...,\nNetgear WNDR3700\n\n\nIEEE 802.11n draft ver...,['https://i.sstatic.net/LvSzZ.jpg']


## Filter

In [12]:
# filter only the images column
cloud_train_img=cloud_train["images"]
device_train_img=device_train["images"]

In [13]:
cloud_train_img.loc[2],device_train_img.loc[2]

("['https://i.sstatic.net/vUGt6.png', 'https://i.sstatic.net/LE4RJ.png']",
 "['https://i.sstatic.net/sGRVz.jpg']")

# OCR 

In [47]:
import requests
from PIL import Image
from io import BytesIO
import ast
from tqdm import tqdm
from tabulate import tabulate

### Img url Cleaning

In [48]:
# mime type setup
def get_mime_type_and_url(image_url):
    # If image_url is a stringified list, parse it and extract the first URL
    if isinstance(image_url, str) and image_url.startswith("["):
        try:
            url_list = ast.literal_eval(image_url)
            image_url = url_list[0] if url_list else ""
        except Exception as e:
            print(f"Failed to parse: {e}")
            return None, None

    # Clean any surrounding quotes or whitespace
    image_url = image_url.strip().strip('"').strip("'")

    # Determine MIME type from file extension
    if image_url.endswith(".png"):
        mime_type = "image/png"
    elif image_url.endswith(".jpg") or image_url.endswith(".jpeg"):
        mime_type = "image/jpeg"
   # elif image_url.endswith(".svg"):
    #    mime_type = "image/svg"
    else:
        print(f"Unsupported format: {image_url}")
        return None, None

    return image_url, mime_type

def get_response(image_url):
    clean_url, mime_type = get_mime_type_and_url(image_url)
    if not clean_url:
        return None, None

    try:
        response = requests.get(clean_url)
        response.raise_for_status()
        content = response.content
     #   print(f"Downloaded: {clean_url}")
        return content, mime_type
    except Exception as e:
        print(f"Download failed for {clean_url}: {e}")
        return None, None


### Layout Setup 

In [49]:
# Combine text pieces from layout into one string
def extract_text(layout, text):
    pieces = []
    for segment in layout.text_anchor.text_segments:
        pieces.append(text[segment.start_index:segment.end_index])
    return ''.join(pieces)

# Get bounding box coordinates and size from layout vertices
def get_bbox(layout):
    xs = [v.x for v in layout.bounding_poly.vertices]
    ys = [v.y for v in layout.bounding_poly.vertices]
    x = min(xs)
    y = min(ys)
    width = max(xs) - x
    height = max(ys) - y
    return x, y, width, height


### Process Img -> Google DOCAI

In [50]:
def process_image_with_docai(image_url):
    content, mime_type = get_response(image_url)
    if not content:
        return None

    try:
        raw_document = documentai.RawDocument(content=content, mime_type=mime_type)
        request = documentai.ProcessRequest(name=processor_name, raw_document=raw_document)
        result = client.process_document(request=request)
        document = result.document
    except Exception as e:
        return {"error": str(e), "image_url": image_url}

    output = {
        "tables": [],
        "non_table_text": []
    }

    table_text_ranges = []

    for page in document.pages:
        for table in page.tables:
            rows = list(table.header_rows) + list(table.body_rows)
            table_data = []

            for row in rows:
                cells = []
                for cell in row.cells:
                    text = extract_text(cell.layout, document.text).strip()
                    cells.append(text)
                    for segment in cell.layout.text_anchor.text_segments:
                        table_text_ranges.append((segment.start_index, segment.end_index))
                table_data.append(cells)

            # No splitting logic, just tabulate the raw table_data
            ascii_table = tabulate(table_data, tablefmt="grid")

            output["tables"].append({
                "content": ascii_table,
                "raw_rows": table_data
            })

        paragraphs = sorted(page.paragraphs, key=lambda p: get_bbox(p.layout)[1])
        for para in paragraphs:
            if not para.layout.text_anchor.text_segments:
                continue

            start = para.layout.text_anchor.text_segments[0].start_index
            end = para.layout.text_anchor.text_segments[-1].end_index

            if any(start <= s < end or s < end <= e for s, e in table_text_ranges):
                continue

            text = extract_text(para.layout, document.text).strip()
            if text:
                x, _, _, _ = get_bbox(para.layout)
                indent = " " * (x // 10)
                output["non_table_text"].append(f"{indent}{text}")

    return output

# Extract Response -> Train

###  Cloud -> Images -> OCR

In [18]:
cloud_train_img.head(2)

0    ['https://i.sstatic.net/D7pQd.png']
1    ['https://i.sstatic.net/Ku2DV.png']
Name: images, dtype: object

In [19]:
#cloud_train_img = [img for img in cloud_train_img if '.svg' not in img.lower()]


In [20]:
cloud_train_img = [img for img in cloud_train_img if '.gif' not in img.lower()]


In [21]:
cloud_train.shape

(4834, 11)

In [22]:
len(cloud_train_img)

4817

In [25]:
"""import ast
import json
from tqdm import tqdm

output_file = "../../data/processed/ocr/cloud/ocr_extracted_cloud_train.jsonl"

with open(output_file, "w", encoding="utf-8") as f_jsonl:  # open once to clear file
    pass

for row_idx, image_cell in enumerate(tqdm(cloud_train_img, desc="Processing images")):
    ocr_outputs = []

    if isinstance(image_cell, str) and image_cell.startswith("["):
        try:
            image_urls = ast.literal_eval(image_cell)
        except Exception as e:
            result = {
                "row": row_idx,
                "ocr": [f"Parsing error: {e}"],
                "image_url": image_cell
            }
            with open(output_file, "a", encoding="utf-8") as f_jsonl:
                f_jsonl.write(json.dumps(result, ensure_ascii=False) + "\n")
            continue
    else:
        image_urls = [image_cell]

    for img_idx, image_url in enumerate(image_urls, start=1):
        doc_output = process_image_with_docai(image_url)

        if doc_output:
            if doc_output["tables"]:
                for t_idx, table in enumerate(doc_output["tables"], start=1):
                    ocr_outputs.append(f"img#{img_idx} - table#{t_idx}:\n{table}")
            if doc_output["non_table_text"]:
                ocr_outputs.append(
                    f"img#{img_idx} - non-table text:\n" + "\n".join(doc_output["non_table_text"])
                )
        else:
            ocr_outputs.append(f"img#{img_idx}: Skipped (invalid or failed OCR)")

    result = {
        "row": row_idx,
        "ocr": ocr_outputs,
        "image_url": image_urls
    }

    # Append the current result immediately after processing each row
    with open(output_file, "a", encoding="utf-8") as f_jsonl:
        f_jsonl.write(json.dumps(result, ensure_ascii=False) + "\n")
"""

'import ast\nimport json\nfrom tqdm import tqdm\n\noutput_file = "cloud-train-img-ocr-response.jsonl"\n\nwith open(output_file, "w", encoding="utf-8") as f_jsonl:  # open once to clear file\n    pass\n\nfor row_idx, image_cell in enumerate(tqdm(cloud_train_img, desc="Processing images")):\n    ocr_outputs = []\n\n    if isinstance(image_cell, str) and image_cell.startswith("["):\n        try:\n            image_urls = ast.literal_eval(image_cell)\n        except Exception as e:\n            result = {\n                "row": row_idx,\n                "ocr": [f"Parsing error: {e}"],\n                "image_url": image_cell\n            }\n            with open(output_file, "a", encoding="utf-8") as f_jsonl:\n                f_jsonl.write(json.dumps(result, ensure_ascii=False) + "\n")\n            continue\n    else:\n        image_urls = [image_cell]\n\n    for img_idx, image_url in enumerate(image_urls, start=1):\n        doc_output = process_image_with_docai(image_url)\n\n        if d

In [24]:
import ast
import json
from tqdm import tqdm

output_file = "../../data/processed/ocr/cloud/ocr_extracted_cloud_train.jsonl"

# Clear the file at start
with open(output_file, "w", encoding="utf-8") as f_jsonl:
    pass

for row_idx, image_cell in enumerate(tqdm(cloud_train_img, desc="Processing images")):
    images = []

    # Parse image URLs
    if isinstance(image_cell, str) and image_cell.startswith("["):
        try:
            image_urls = ast.literal_eval(image_cell)
        except Exception as e:
            images.append({
                "image_url": image_cell,
                "response": [f"Parsing error: {e}"]
            })
            result = {
                "row": row_idx,
                "total_images": len(images),
                "images": images
            }
            with open(output_file, "a", encoding="utf-8") as f_jsonl:
                f_jsonl.write(json.dumps(result, ensure_ascii=False) + "\n")
            continue
    else:
        image_urls = [image_cell]

    for image_url in image_urls:
        responses = []
        doc_output = process_image_with_docai(image_url)

        if doc_output:
            if doc_output.get("tables"):
                for t_idx, table in enumerate(doc_output["tables"], start=1):
                    responses.append({
                        "table#": t_idx,
                        "table": table
                    })
            if doc_output.get("non_table_text"):
                responses.append({
                    "non_table_text": doc_output["non_table_text"]
                })
        else:
            responses.append("Skipped (invalid or failed OCR)")

        images.append({
            "image_url": image_url,
            "response": responses
        })

    result = {
        "row": row_idx,
        "total_images": len(images),
        "images": images
    }

    # Write result after processing each row
    with open(output_file, "a", encoding="utf-8") as f_jsonl:
        f_jsonl.write(json.dumps(result, ensure_ascii=False) + "\n")


Processing images:   1%|          | 29/4817 [02:46<7:39:18,  5.76s/it] 


KeyboardInterrupt: 

In [111]:
cloud_train_img[44]

"['https://cdn.rawgit.com/buildkite/cloudformation-launch-stack-button-svg/master/launch-stack.svg']"

In [107]:
ocr_outputs[:2]

["img#1 - table#1:\n{'content': '+-------------------+----------+-------+--------+\\n| Security group ID | Protocol | Ports | Source |\\n+-------------------+----------+-------+--------+\\n| sg-               | All      | All   | sg-    |\\n+-------------------+----------+-------+--------+', 'raw_rows': [['Security group ID', 'Protocol', 'Ports', 'Source'], ['sg-', 'All', 'All', 'sg-']]}",
 'img#1 - non-table text:\n    Inbound rules\n                    Outbound rules\n                                                                                              vpc endpoint\n                                                                                                                           < 1 >\n                                                                                              security group']

In [117]:
import json
import ast

output_file = "../../data/processed/ocr/cloud/ocr_extracted_cloud_train.jsonl"

print("\n=========== Responses ===========\n")

with open(output_file, "r", encoding="utf-8") as f_jsonl:
    for line in f_jsonl:
        result = json.loads(line)
        print(f"Row {result['row']}")
        print("-" * 30)

        for ocr_output in result["ocr"]:
            if "table#" in ocr_output:
                try:
                    dict_str = ocr_output.split(":", 1)[1].strip()
                    table_dict = ast.literal_eval(dict_str)
                    print(f"{ocr_output.split(':', 1)[0]}:")
                    print(table_dict.get("content", "No content found"))
                except Exception:
                    print(ocr_output)
            else:
                print(ocr_output)
            print()  # blank line between OCR outputs

        print("Image URL(s):")
        for url in result["image_url"]:
            print(f"  - {url}")

        print("\n" + "=" * 50 + "\n")  # Separator between rows



=========== Responses ===========

Row 0
------------------------------
img#1 - table#1:
+----------------------------+-------------------------+-----------------------------------------------+-------------------+------------------------+------------------+-------------------------------------------+-----------------+-----------------+-----------------------------------------------------+---------------------------------------------------------------------+------------+--------------+
| vCPU: Up to 2vCPUs         | Memory: 8.0 GiB         | Network: Low to Moderate                      | Max EN Max ENI: 4 | vCPU: 4 vCPUs          | Memory: 16.0 GIB | Network: Up to 10 Gbps                    | Max ENI: 4      | r5.2xlarge      | Memory: 64.0 GIB                                    | Network: 10 Gbps                                                    | Max ENI: 4 | Max IPs: 60  |
| Max IPs: 36                | Memory: 16.0 GIB        | Network: Moderate                             | Max

### Device -> Images -> OCR

In [30]:
device_train_img = [img for img in device_train_img if '.svg' not in img.lower()]


In [32]:
len(device_train_img)

3079

In [31]:
device_train.shape

(3081, 11)

In [ ]:
import ast
import json
from tqdm import tqdm

results = []

for row_idx, image_cell in enumerate(tqdm(cloud_train_img[:30], desc="Processing images")):
    ocr_outputs = []

    # Parse stringified list or treat as a single URL
    if isinstance(image_cell, str) and image_cell.startswith("["):
        try:
            image_urls = ast.literal_eval(image_cell)
        except Exception as e:
            results.append({
                "row": row_idx,
                "ocr": [f"Parsing error: {e}"],
                "image_url": image_cell
            })
            continue
    else:
        image_urls = [image_cell]

    for img_idx, image_url in enumerate(image_urls, start=1):
        doc_output = process_image_with_docai(image_url)

        if doc_output:
            if doc_output["tables"]:
                for t_idx, table in enumerate(doc_output["tables"], start=1):
                    ocr_outputs.append(f"img#{img_idx} - table#{t_idx}:\n{table}")
            if doc_output["non_table_text"]:
                ocr_outputs.append(
                    f"img#{img_idx} - non-table text:\n" + "\n".join(doc_output["non_table_text"])
                )
        else:
            ocr_outputs.append(f"img#{img_idx}: Skipped (invalid or failed OCR)")

    results.append({
        "row": row_idx,
        "ocr": ocr_outputs,
        "image_url": image_urls
    })

# Save results as JSON Lines file
with open("../../data/processed/ocr/device/ocr_extracted_device_train.jsonl", "w", encoding="utf-8") as f_jsonl:
    for entry in results:
        f_jsonl.write(json.dumps(entry, ensure_ascii=False) + "\n")


# Extract Response -> Test

### Data load 

In [3]:
import pandas as pd 
import os 


# testset

cloud_directory=r"D:\Research2-Logs-Qa\multi-modal-log-qa\data\vlm\processed\multimodal-cloud-dataset"
device_directory=r"D:\Research2-Logs-Qa\multi-modal-log-qa\data\vlm\processed\multimodal-device-dataset"

cloud_test_directory=os.path.join(cloud_directory, "../../data/splits/cloud/cloud_test.xlsx")
device_test_directory=os.path.join(device_directory, "../../data/splits/device/device_test.xlsx")
# Read the CSV files
cloud_test=pd.read_csv(cloud_test_directory)
device_test=pd.read_csv(device_test_directory)

In [9]:
cloud_test.shape,device_test.shape

((1568, 13), (947, 12))

In [12]:
# filter only the images column
cloud_test_img=cloud_test["images"]
device_test_img=device_test["images"]

In [18]:
cloud_test_img[1023]

"['https://i.sstatic.net/quzpi.jpg', 'https://i.sstatic.net/GXv8m.png', 'https://i.sstatic.net/9bIY0.png']"

###  Cloud -> Images -> OCR

In [20]:
cloud_test_img.head(2)

0                  ['https://i.sstatic.net/QQmRh.png']
1    ['https://i.sstatic.net/VJ1po.png', 'https://i...
Name: images, dtype: object

In [21]:
#cloud_train_img = [img for img in cloud_train_img if '.svg' not in img.lower()]


In [23]:
#cloud_test_img = [img for img in cloud_train_img if '.gif' not in img.lower()]


In [25]:
len(cloud_test_img)

1568

In [52]:
import ast
import json
from tqdm import tqdm

output_file = "../../data/processed/ocr/cloud/ocr_extracted_cloud_test.jsonl"
failed_file = "cloud-test-img-ocr-failed.jsonl"

# Clear both files at start
for f in (output_file, failed_file):
    with open(f, "w", encoding="utf-8"):
        pass

def log_failure(row_idx: int, image_url: str, reason: str) -> None:
    """Write one failed URL as JSONL to failed_file."""
    failure_record = {
        "row": row_idx,
        "image_url": image_url,
        "reason": reason
    }
    with open(failed_file, "a", encoding="utf-8") as f_fail:
        f_fail.write(json.dumps(failure_record, ensure_ascii=False) + "\n")

for row_idx, image_cell in enumerate(tqdm(cloud_test_img, desc="Processing images")):
    images = []

    # Parse image URLs
    if isinstance(image_cell, str) and image_cell.startswith("["):
        try:
            image_urls = ast.literal_eval(image_cell)
        except Exception as e:
            err_msg = f"Parsing error: {e}"
            log_failure(row_idx, image_cell, err_msg)
            # Still record the error in the main file for completeness
            images.append({
                "image_url": image_cell,
                "response": [err_msg]
            })
            result = {
                "row": row_idx,
                "total_images": len(images),
                "images": images
            }
            with open(output_file, "a", encoding="utf-8") as f_jsonl:
                f_jsonl.write(json.dumps(result, ensure_ascii=False) + "\n")
            continue
    else:
        image_urls = [image_cell]

    for image_url in image_urls:
        responses = []
        doc_output = process_image_with_docai(image_url)

        if doc_output:
            if doc_output.get("tables"):
                for t_idx, table in enumerate(doc_output["tables"], start=1):
                    responses.append({"table#": t_idx, "table": table})
            if doc_output.get("non_table_text"):
                responses.append({"non_table_text": doc_output["non_table_text"]})
        else:
            err_msg = "Skipped (invalid or failed OCR)"
            log_failure(row_idx, image_url, err_msg)
            responses.append(err_msg)

        images.append({
            "image_url": image_url,
            "response": responses
        })

    result = {
        "row": row_idx,
        "total_images": len(images),
        "images": images
    }

    with open(output_file, "a", encoding="utf-8") as f_jsonl:
        f_jsonl.write(json.dumps(result, ensure_ascii=False) + "\n")


Processing images:   0%|          | 0/1568 [00:00<?, ?it/s]

Processing images:  36%|███▋      | 571/1568 [1:20:44<2:20:59,  8.48s/it]  


KeyboardInterrupt: 

In [ ]:
cloud_train_img[44]

"['https://cdn.rawgit.com/buildkite/cloudformation-launch-stack-button-svg/master/launch-stack.svg']"

In [ ]:
ocr_outputs[:2]

["img#1 - table#1:\n{'content': '+-------------------+----------+-------+--------+\\n| Security group ID | Protocol | Ports | Source |\\n+-------------------+----------+-------+--------+\\n| sg-               | All      | All   | sg-    |\\n+-------------------+----------+-------+--------+', 'raw_rows': [['Security group ID', 'Protocol', 'Ports', 'Source'], ['sg-', 'All', 'All', 'sg-']]}",
 'img#1 - non-table text:\n    Inbound rules\n                    Outbound rules\n                                                                                              vpc endpoint\n                                                                                                                           < 1 >\n                                                                                              security group']

In [ ]:
import json
import ast

output_file = "../../data/processed/ocr/cloud/ocr_extracted_cloud_train.jsonl"

print("\n=========== Responses ===========\n")

with open(output_file, "r", encoding="utf-8") as f_jsonl:
    for line in f_jsonl:
        result = json.loads(line)
        print(f"Row {result['row']}")
        print("-" * 30)

        for ocr_output in result["ocr"]:
            if "table#" in ocr_output:
                try:
                    dict_str = ocr_output.split(":", 1)[1].strip()
                    table_dict = ast.literal_eval(dict_str)
                    print(f"{ocr_output.split(':', 1)[0]}:")
                    print(table_dict.get("content", "No content found"))
                except Exception:
                    print(ocr_output)
            else:
                print(ocr_output)
            print()  # blank line between OCR outputs

        print("Image URL(s):")
        for url in result["image_url"]:
            print(f"  - {url}")

        print("\n" + "=" * 50 + "\n")  # Separator between rows



=========== Responses ===========

Row 0
------------------------------
img#1 - table#1:
+----------------------------+-------------------------+-----------------------------------------------+-------------------+------------------------+------------------+-------------------------------------------+-----------------+-----------------+-----------------------------------------------------+---------------------------------------------------------------------+------------+--------------+
| vCPU: Up to 2vCPUs         | Memory: 8.0 GiB         | Network: Low to Moderate                      | Max EN Max ENI: 4 | vCPU: 4 vCPUs          | Memory: 16.0 GIB | Network: Up to 10 Gbps                    | Max ENI: 4      | r5.2xlarge      | Memory: 64.0 GIB                                    | Network: 10 Gbps                                                    | Max ENI: 4 | Max IPs: 60  |
| Max IPs: 36                | Memory: 16.0 GIB        | Network: Moderate                             | Max